        # إنتاج نور باستخدام SadTalker على Kaggle GPU

        هذا الدفتر يستخدم GPU المجاني في Kaggle لتشغيل مسار `SadTalkerProvider` الموجود في مستودع `facebook-ai-studio`. ابدأ دائمًا بـShort قصير، ثم شغّل الحلقة الكاملة بعد مراجعة الناتج.

        يتطلب التشغيل إضافة `FISH_API_KEY` و`FISH_VOICE_ID` إلى Kaggle Secrets يدويًا إذا أردت أن يولّد الدفتر الصوت عبر Fish Audio. لا تكتب المفاتيح داخل أي خلية أو ملف. Kaggle يوفّر GPU مجانيًا بحصة أسبوعية متغيرة، لذلك قد لا يكون متاحًا في كل محاولة [1].
        

In [ ]:
        import os
        import sys
        import subprocess
        from pathlib import Path
        from IPython.display import Video, FileLink, display

        print(sys.version)
        subprocess.run([
            "nvidia-smi", "--query-gpu=name,memory.total,driver_version",
            "--format=csv,noheader",
        ], check=True)
        import torch
        print("PyTorch:", torch.__version__)
        print("CUDA available:", torch.cuda.is_available())
        if not torch.cuda.is_available():
            raise RuntimeError("Kaggle GPU is not enabled. Open Notebook options and select Accelerator = GPU.")
        print("GPU:", torch.cuda.get_device_name(0))
        

        ## تنزيل المشروع وSadTalker

        يُعاد تنزيل الملفات داخل جلسة Kaggle المؤقتة عند بدء جلسة جديدة. النموذج لا يحتاج إلى أي مفتاح GitHub لأن المستودعين عامّان.
        

In [ ]:
        import shutil

        workdir = Path("/kaggle/working")
        project = workdir / "facebook-ai-studio"
        sadtalker = workdir / "SadTalker"
        if project.exists():
            shutil.rmtree(project)
        if sadtalker.exists():
            shutil.rmtree(sadtalker)
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ahmedadelmiedo-hub/facebook-ai-studio.git", str(project)], check=True)
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/OpenTalker/SadTalker.git", str(sadtalker)], check=True)
        reference = project / "storage/references/nour-master-v1.png"
        assert reference.is_file(), reference
        print("Project:", project)
        print("SadTalker:", sadtalker)
        print("Reference:", reference)
        

        ## تثبيت اعتماديات SadTalker

        إصدارات Kaggle قد تختلف، لذلك تُستخدم حزم متوافقة مع PyTorch الموجود في الجلسة. إذا ظهر تعارض في حزمة واحدة، أعد تشغيل الجلسة ثم نفّذ الخلية مرة أخرى قبل تغيير الإصدارات.
        

In [ ]:
        packages = [
            "numpy<2",
            "scipy<1.12",
            "scikit-image>=0.21,<0.23",
            "librosa>=0.10,<0.11",
            "numba",
            "resampy",
            "pydub",
            "kornia",
            "tqdm",
            "yacs",
            "pyyaml",
            "joblib",
            "basicsr==1.4.2",
            "facexlib==0.3.0",
            "gfpgan",
            "av",
            "safetensors",
            "face_alignment==1.3.5",
        ]
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(project / "requirements.txt")], check=True)
        subprocess.run(["bash", str(sadtalker / "scripts/download_models.sh")], check=True)
        print("Dependencies and checkpoints prepared.")
        

        ## تحميل أسرار Fish Audio من Kaggle Secrets

        أضف السرّين من واجهة Kaggle تحت **Add-ons → Secrets** بأسماء `FISH_API_KEY` و`FISH_VOICE_ID`. الخلية لا تطبع القيم. إذا كان لديك ملف صوتي جاهز، يمكن تجاوز هذه الخلية وتشغيل SadTalker مباشرةً بدل autopilot.
        

In [ ]:
        from kaggle_secrets import UserSecretsClient

        secrets = UserSecretsClient()
        for key in ("FISH_API_KEY", "FISH_VOICE_ID"):
            value = secrets.get_secret(key).strip()
            if not value:
                raise RuntimeError(f"Missing Kaggle Secret: {key}")
            os.environ[key] = value
        print("Fish Audio secrets loaded without printing their values.")
        

        ## دالة التشغيل الموحدة

        تستخدم هذه الدالة نفس `core.autopilot` الموجود في GitHub، وتضبط SadTalker على GPU تلقائيًا. لا تُمرّر أي مفتاح داخل سطر الأمر؛ الصوت يُقرأ من متغير البيئة.
        

In [ ]:
        def run_avatar_episode(script_file, episode_name, output_dir, max_characters):
            env = os.environ.copy()
            env.update({
                "SADTALKER_ROOT": str(sadtalker),
                "SADTALKER_PYTHON": sys.executable,
                "AVATAR_PROVIDER": "sadtalker",
                "AVATAR_SOURCE_IMAGE": str(reference),
                "SADTALKER_CPU": "0",
                "SADTALKER_ENHANCER": "",
                "SADTALKER_STILL": "0",
                "PYTHONPATH": str(project),
            })
            command = [
                sys.executable,
                "-m", "core.autopilot",
                "--avatar-episode",
                "--avatar-provider", "sadtalker",
                "--avatar-source-image", str(reference),
                "--script-file", str(project / script_file),
                "--voice", os.environ["FISH_VOICE_ID"],
                "--format", "long",
                "--episode-name", episode_name,
                "--output-dir", str(project / output_dir),
                "--character-file", str(project / "content/characters/nour-podcast-host-v1.json"),
                "--scene-prompt", "Nour narrates the case from her podcast studio",
                "--scene-camera", "cinematic medium shot with microphone and evidence board",
                "--avatar-max-characters", str(max_characters),
                "--review-status", "approved",
                "--keep-audio",
            ]
            subprocess.run(command, cwd=project, env=env, check=True)
            outputs = sorted((project / output_dir).glob("*.mp4"))
            if not outputs:
                raise FileNotFoundError(f"No MP4 found in {project / output_dir}")
            return outputs[0]
        

        ## اختبار Short أولًا

        هذه الخلية تنتج Short قصيرًا من `case-001-EP02-HOOK.txt`. لا تنتقل إلى الحلقة الكاملة إلا بعد فحص الفيديو والصوت وCaptions.
        

In [ ]:
        short_video = run_avatar_episode(
            "content/scripts/shorts/case-001-EP02-HOOK.txt",
            "case-001-EP02-SADTALKER-KAGGLE-SHORT",
            "storage/autopilot/avatar-sadtalker-kaggle-short",
            240,
        )
        print(short_video, short_video.stat().st_size)
        subprocess.run([
            "ffprobe", "-v", "error", "-show_entries", "format=duration",
            "-show_entries", "stream=codec_name,width,height",
            "-of", "default=noprint_wrappers=1", str(short_video),
        ], check=True)
        display(Video(str(short_video), embed=True))
        

        ## إنتاج الحلقة الكاملة EP02

        شغّل هذه الخلية فقط بعد نجاح Short. الناتج سيكون حلقة أفقية طويلة مع Captions عربية مدمجة. قد تنتهي جلسة Kaggle قبل اكتمال الحلقة إذا كانت الجلسة أو الحصة الأسبوعية محدودة؛ في هذه الحالة أعد تشغيل الجلسة والخلية من البداية.
        

In [ ]:
        full_video = run_avatar_episode(
            "content/scripts/long/case-001-EP02.txt",
            "case-001-EP02-SADTALKER-KAGGLE-FULL",
            "storage/autopilot/avatar-sadtalker-kaggle-full",
            900,
        )
        print(full_video, full_video.stat().st_size)
        subprocess.run([
            "ffprobe", "-v", "error", "-show_entries", "format=duration",
            "-show_entries", "stream=codec_name,width,height",
            "-of", "default=noprint_wrappers=1", str(full_video),
        ], check=True)
        display(Video(str(full_video), embed=True))
        

        ## تنزيل الناتج

        استخدم الرابط التالي لتنزيل الحلقة من جلسة Kaggle. لا تُنشر الحلقة قبل فحصها يدويًا، وأضف إفصاحًا مناسبًا عن كون المذيعة والشكل مولدين بالذكاء الاصطناعي.
        

In [ ]:
        display(FileLink(str(full_video)))
        